In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [4]:
# ---------- 1. 读取数据 ----------
base = pd.read_csv('../Dataset/ml-100k/u1.base', sep='\t',
                   names=['user_id', 'item_id', 'rate', 'timestamp'])
test = pd.read_csv('../Dataset/ml-100k/u1.test', sep='\t',
                   names=['user_id', 'item_id', 'rate', 'timestamp'])

n_users = base['user_id'].max()   # 943
n_items = base['item_id'].max()   # 1682


In [5]:
# ---------- 2. 构建评分矩阵和掩码 ----------
R = np.zeros((n_users, n_items), dtype=np.float32)
M = np.zeros((n_users, n_items), dtype=np.float32)

for row in base.itertuples():
    u = row.user_id - 1
    i = row.item_id - 1
    R[u, i] = row.rate
    M[u, i] = 1.0

R_tensor = torch.from_numpy(R)
M_tensor = torch.from_numpy(M)


In [6]:
# ---------- 3. 定义模型 ----------
class AutoRec(nn.Module):
    def __init__(self, num, hidden_dim):
        super(AutoRec, self).__init__()
        self.encoder = nn.Linear(num, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, num)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        h = self.sigmoid(self.encoder(x))
        r_hat = self.decoder(h)
        return r_hat

model = AutoRec(num=n_items, hidden_dim=200)

In [7]:
# ---------- 4. 损失函数 ----------
def masked_mse_loss(pred, target, mask, V, W, lam):
    diff = (pred - target) * mask
    mse = (diff ** 2).sum() / mask.sum()
    reg = lam * (V.norm()**2 + W.norm()**2) / 2
    return mse + reg

In [ ]:
# # ---------- 5. 训练 ----------
# lamda = 0.001
# optimizer = torch.optim.Adam(model.parameters(), lamda)
# num_epochs = 1000

# for epoch in range(num_epochs):
#     model.train()
#     optimizer.zero_grad()
    
#     pred = model(R_tensor)
#     loss = masked_mse_loss(
#         pred, R_tensor, M_tensor,
#         model.encoder.weight, model.decoder.weight,
#         lamda
#     )
#     loss.backward()
#     optimizer.step()
    
#     if (epoch + 1) % 20 == 0:
#         print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 20, Loss: 1.7260
Epoch 40, Loss: 1.4421
Epoch 60, Loss: 1.2983
Epoch 80, Loss: 1.2533
Epoch 100, Loss: 1.2245
Epoch 120, Loss: 1.1996
Epoch 140, Loss: 1.1760
Epoch 160, Loss: 1.1528
Epoch 180, Loss: 1.1309
Epoch 200, Loss: 1.1101
Epoch 220, Loss: 1.0909
Epoch 240, Loss: 1.0733
Epoch 260, Loss: 1.0570
Epoch 280, Loss: 1.0424
Epoch 300, Loss: 1.0291
Epoch 320, Loss: 1.0166
Epoch 340, Loss: 1.0052
Epoch 360, Loss: 0.9945
Epoch 380, Loss: 0.9845
Epoch 400, Loss: 0.9751
Epoch 420, Loss: 0.9658
Epoch 440, Loss: 0.9564
Epoch 460, Loss: 0.9475
Epoch 480, Loss: 0.9383
Epoch 500, Loss: 0.9297
Epoch 520, Loss: 0.9213
Epoch 540, Loss: 0.9133
Epoch 560, Loss: 0.9058
Epoch 580, Loss: 0.8985
Epoch 600, Loss: 0.8914
Epoch 620, Loss: 0.8845
Epoch 640, Loss: 0.8777
Epoch 660, Loss: 0.8711
Epoch 680, Loss: 0.8645
Epoch 700, Loss: 0.8583
Epoch 720, Loss: 0.8522
Epoch 740, Loss: 0.8462
Epoch 760, Loss: 0.8403
Epoch 780, Loss: 0.8343
Epoch 800, Loss: 0.8284
Epoch 820, Loss: 0.8226
Epoch 840, Loss: 0.8

In [ ]:
# # ---------- 6. 提取训练后的参数 ----------
# W   = model.encoder.weight   # (hidden_dim, n_items)
# b_z = model.encoder.bias   # (hidden_dim,)
# V   = model.decoder.weight  # (n_items, hidden_dim)
# b   = model.decoder.bias     # (n_items,)

# print("W shape:  ", W.shape)     # (200, 1682)
# print("b^z shape:", b_z.shape)   # (200,)
# print("V shape:  ", V.shape)     # (1682, 200)
# print("b shape:  ", b.shape)     # (1682,)

W shape:   torch.Size([200, 1682])
b^z shape: torch.Size([200])
V shape:   torch.Size([1682, 200])
b shape:   torch.Size([1682])


In [ ]:
# import os
# print(os.getcwd())
# torch.save(model.state_dict(), 'AutoRec.pt')

d:\Code\Recommender Systems\Review


In [8]:
model = AutoRec(num=n_items, hidden_dim=200)
model.load_state_dict(torch.load('AutoRec.pt'))


<All keys matched successfully>

In [9]:
# ---------- 7. 测试集评估 ----------
model.eval()
with torch.no_grad():
    R_pred = model(R_tensor).detach().numpy()

errors = []
for row in test.itertuples():
    u = row.user_id - 1
    i = row.item_id - 1
    errors.append((R_pred[u, i] - row.rate) ** 2)

rmse = np.sqrt(np.mean(errors))
print(f"Test RMSE: {rmse:.4f}")

Test RMSE: 0.9547


┌─────────────────────────────────────────────────────┐
│  ⑤ optimizer.zero_grad()   清空旧梯度                │
└─────────────────────────────────────────────────────┘
                        ↓
┌─────────────────────────────────────────────────────┐
│  ⑥ 前向传播  pred = model(R_tensor)                  │
│     R → encoder → h → decoder → pred                │
│     （同时构建计算图）                                │
└─────────────────────────────────────────────────────┘
                        ↓
┌─────────────────────────────────────────────────────┐
│  ⑦ 计算损失  loss = masked_mse_loss(...)             │
│     loss 是标量，是计算图的终点                       │
└─────────────────────────────────────────────────────┘
                        ↓
┌─────────────────────────────────────────────────────┐
│  ⑧ 反向传播  loss.backward()                         │
│     沿计算图反向，链式法则求梯度                      │
│     梯度写入 V.grad, b_z.grad, W.grad, b.grad        │
└─────────────────────────────────────────────────────┘
                        ↓
┌─────────────────────────────────────────────────────┐
│  ⑨ 更新参数  optimizer.step()                        │
│     Adam 根据梯度修改 V, b_z, W, b 的值              │
└─────────────────────────────────────────────────────┘
                        ↓
                  进入下一个 epoch

In [ ]:
import matplotlib
matplotlib.use('Agg')         
import matplotlib.pyplot as plt

epochs = [500, 1000, 2000]
data = {
    '0.0001': [1.0856, 1.0572, 1.0073],
    '0.001':  [0.9738, 0.9563, 0.9683],
    '0.01':   [1.0186, 1.0161, 1.0151],
    '0.1':    [1.0377, 1.0375, 1.0361],
}
        
plt.figure(figsize=(8, 5))
for lam, rmses in data.items():
    plt.plot(epochs, rmses, marker='o', label=f'lam={lam}')
plt.xlabel('Epoch')
plt.ylabel('Test RMSE')
plt.title('Test RMSE vs Epoch for Different lam')
plt.legend()
plt.grid(True)
plt.savefig('autorec_rmse.png', dpi=150, bbox_inches='tight')
plt.close()
print("图片已保存")

图片已保存
